# EDA - Medical Text Classification Dataset

Dataset com **dois campos**:
- `target`: categoria clínica de **1 a 5** (não representa gravidade)
- `text`: abstract médico no schema canônico do projeto

**Objetivos:**
1. Entender distribuição e balanceamento das 5 classes
2. Analisar características dos textos por classe (tamanho, vocabulário)
3. Identificar termos discriminativos por label via TF-IDF
4. Detectar problemas: nulos, duplicados, desbalanceamento

## 1. Imports e Configurações

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from triage_ml.data.prepare import prepare_dataset
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
colors = sns.color_palette('Set2', 5)

DATA_PATH = PROJECT_ROOT / 'data' / 'medical_tc_train.csv'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
LABEL_NAMES = {
    1: 'Neoplasias',
    2: 'Digestivo/Infeccioso',
    3: 'Doenças do sistema nervoso',
    4: 'Cardiovascular',
    5: 'Condições patológicas gerais'
}

print("✓ Imports carregados")

## 2. Carregamento e Inspeção Inicial

In [ ]:
raw_df = pd.read_csv(DATA_PATH)
df, preparation_report = prepare_dataset(raw_df, sample_size=5000, random_state=42)
print(f"Relatório de preparação: {preparation_report}")
print(f"Shape: {df.shape}")
print(f"Colunas: {df.columns.tolist()}")
print(f"\nTipos:\n{df.dtypes}")
print(f"\nValores nulos:\n{df.isnull().sum()}")
print(f"\nTextos duplicados após preparação: {df['text'].duplicated().sum()}")
print(f"Labels únicos: {sorted(df['target'].unique())}")

# Privacidade operacional: nunca imprimir abstracts no notebook versionado.
# Apenas agregados (n_amostras, n_chars, n_palavras) são emitidos por outras células.
print(f"Amostras no recorte: {len(df)} | colunas canônicas: {list(df.columns)}")


## 3. Distribuição de Classes (Balanceamento)

In [ ]:
label_counts = df['target'].value_counts().sort_index()
label_pct = (label_counts / len(df) * 100).round(2)

print("Distribuição de labels:")
print(pd.DataFrame({'Count': label_counts, 'Percentual (%)': label_pct}))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].bar(label_counts.index, label_counts.values, color=colors)
axes[0].set_title('Contagem por Label (1–5)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Condition Label')
axes[0].set_ylabel('Frequência')
for bar, pct in zip(bars, label_pct.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{pct}%', ha='center', fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

axes[1].pie(label_counts.values,
            labels=[f'L{i}: {LABEL_NAMES[i]}' for i in label_counts.index],
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Proporção por Label', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_label_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

max_ratio = label_counts.max() / label_counts.min()
print(f"\nRatio máximo/mínimo entre classes: {max_ratio:.2f}x")
if max_ratio > 2:
    print("⚠️  Dataset DESBALANCEADO — considerar class_weight='balanced' ou oversampling")
else:
    print("✓ Dataset relativamente BALANCEADO")
print("✓ Salvo: reports/figures/01_label_distribution.png")

## 4. Análise Estatística dos Textos

In [ ]:
df['char_count'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()
df['sentence_count'] = df['text'].str.count(r'[.!?]+')
df['avg_word_length'] = df['text'].apply(
    lambda x: round(np.mean([len(w) for w in str(x).split()]), 2) if pd.notna(x) else 0
)

stats_cols = ['char_count', 'word_count', 'sentence_count', 'avg_word_length']
print("=== Estatísticas globais dos textos ===")
print(df[stats_cols].describe().round(2))

print("\n=== Média por Label ===")
print(df.groupby('target')[stats_cols].mean().round(2))

## 5. Distribuição de Comprimento por Label

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics = [('word_count', 'Palavras'), ('char_count', 'Caracteres'),
           ('sentence_count', 'Sentenças'), ('avg_word_length', 'Comp. Médio Palavra')]

for ax, (col, label) in zip(axes.flatten(), metrics):
    data = [df[df['target'] == lbl][col].values
            for lbl in sorted(df['target'].unique())]
    bp = ax.boxplot(data, tick_labels=[f'L{i}' for i in sorted(df['target'].unique())],
                    patch_artist=True)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(f'{label} por Label', fontsize=12, fontweight='bold')
    ax.set_xlabel('Condition Label')
    ax.set_ylabel(label)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Distribuição de Métricas de Texto por Label', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_text_stats_by_label.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Salvo: reports/figures/02_text_stats_by_label.png")

## 6. Palavras Mais Frequentes por Label

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 6))
fig.suptitle('Top 15 Palavras por Label (sem stopwords)', fontsize=14, fontweight='bold')

for i, label in enumerate(sorted(df['target'].unique())):
    texts = df[df['target'] == label]['text'].tolist()
    vec = CountVectorizer(stop_words='english', max_features=15)
    matrix = vec.fit_transform(texts)
    word_freq = dict(zip(vec.get_feature_names_out(), matrix.toarray().sum(axis=0)))
    top_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)[:15]
    words, freqs = zip(*top_words)

    axes[i].barh(range(len(words)), freqs, color=colors[i])
    axes[i].set_yticks(range(len(words)))
    axes[i].set_yticklabels(words, fontsize=9)
    axes[i].set_title(f'Label {label}\n{LABEL_NAMES[label]}', fontsize=10, fontweight='bold')
    axes[i].invert_yaxis()
    axes[i].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_top_words_per_label.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Salvo: reports/figures/03_top_words_per_label.png")

## 7. Termos Discriminativos (TF-IDF por Classe)

In [ ]:
# TF-IDF médio por classe — mostra termos mais CARACTERÍSTICOS de cada label
fig, axes = plt.subplots(1, 5, figsize=(22, 6))
fig.suptitle('Top 10 Termos TF-IDF por Label (Discriminativos)', fontsize=14, fontweight='bold')

for i, label in enumerate(sorted(df['target'].unique())):
    texts = df[df['target'] == label]['text'].tolist()
    vec = TfidfVectorizer(stop_words='english', max_features=500, ngram_range=(1, 1))
    tfidf_matrix = vec.fit_transform(texts)
    mean_tfidf = tfidf_matrix.mean(axis=0).A1
    top_idx = mean_tfidf.argsort()[-10:][::-1]
    top_terms = [(vec.get_feature_names_out()[j], mean_tfidf[j]) for j in top_idx]
    words, scores = zip(*top_terms)

    axes[i].barh(range(len(words)), scores, color=colors[i])
    axes[i].set_yticks(range(len(words)))
    axes[i].set_yticklabels(words, fontsize=9)
    axes[i].set_title(f'Label {label}\n{LABEL_NAMES[label]}', fontsize=10, fontweight='bold')
    axes[i].invert_yaxis()
    axes[i].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_tfidf_per_label.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Salvo: reports/figures/04_tfidf_per_label.png")

## 8. Diversidade de Vocabulário (TTR)

In [ ]:
# Type-Token Ratio (TTR) = vocab único / total de palavras — mede riqueza lexical
vocab_stats = []
for label in sorted(df['target'].unique()):
    texts = df[df['target'] == label]['text'].tolist()
    all_words = ' '.join(texts).lower().split()
    unique_words = set(all_words)
    vocab_stats.append({
        'Label': label,
        'Categoria': LABEL_NAMES[label],
        'Amostras': len(texts),
        'Total palavras': len(all_words),
        'Vocabulário único': len(unique_words),
        'TTR': round(len(unique_words) / len(all_words), 4)
    })

vocab_df = pd.DataFrame(vocab_stats)
print("=== Diversidade de Vocabulário por Label ===")
print(vocab_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(vocab_df['Label'], vocab_df['TTR'], color=colors)
ax.set_title('Riqueza Lexical (TTR) por Label', fontsize=13, fontweight='bold')
ax.set_xlabel('Condition Label')
ax.set_ylabel('Type-Token Ratio (0–1)')
ax.set_ylim(0, max(vocab_df['TTR']) * 1.2)
for i, (lbl, ttr) in enumerate(zip(vocab_df['Label'], vocab_df['TTR'])):
    ax.text(lbl, ttr + 0.002, f'{ttr:.3f}', ha='center', fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_lexical_richness.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Salvo: reports/figures/05_lexical_richness.png")

## 9. Validação do Recorte por Label

In [ ]:
# Não imprimir abstracts: as evidências versionadas contêm somente agregados.
validation_by_label = df.groupby('target').agg(
    samples=('text', 'size'),
    min_words=('word_count', 'min'),
    median_words=('word_count', 'median'),
    max_words=('word_count', 'max'),
)
validation_by_label['category'] = validation_by_label.index.map(LABEL_NAMES)
print(validation_by_label.to_string())

## 10. Resumo e Insights para Modelagem

### Características do Dataset
- **Tipo**: Classificação multiclasse de texto médico (5 classes)
- **Target**: `target` (1–5) — categoria clínica, sem ordem de gravidade

### Pontos de Atenção para o Modelo
1. **Balanceamento**: Se desbalanceado → usar `class_weight='balanced'` na Logistic Regression
2. **Comprimento dos textos**: Abstracts longos → TF-IDF com `max_features` adequado; modelos BERT truncam em 512 tokens
3. **Vocabulário médico especializado**: Stop words do sklearn removem termos comuns; termos técnicos são os mais discriminativos
4. **Bigramas**: Expressões como `coronary artery`, `lymph node` mais informativas que unigramas → `ngram_range=(1,2)`
5. **TTR**: Labels com TTR alto têm vocabulário mais diverso → podem ser mais difíceis de classificar

### Próximos Passos
- ✅ Recorte canônico reproduzível e sem conflitos/duplicatas exatas
- ✅ Proveniência, licença e decisão documentadas em `docs/dataset.md`
- ▶ Implementar baseline TF-IDF + Logistic Regression (`02_model_baseline.ipynb`)
- ▶ Extrair pré-processamento para `src/triage_ml/data/`